In [ ]:
from __future__ import annotations

import os
import traceback
from dataclasses import dataclass, field
from pathlib import Path
from typing import Dict, Optional, Tuple, List

import numpy as np
import pandas as pd
import tifffile

from cellpose import models
from scipy.ndimage import gaussian_laplace
from scipy import ndimage as ndi
from scipy.stats import norm
from skimage import filters, morphology
from skimage.filters import threshold_otsu
from skimage.measure import regionprops_table

# Helpers: channels + paths
def get_channel_indices(row: pd.Series, n_channels: int = 4) -> Dict[str, int]:
    """Map channel name -> channel index based on columns ch0..ch{n-1}."""
    channel_map: Dict[str, int] = {}
    for i in range(n_channels):
        name = row.get(f"ch{i}")
        if isinstance(name, str) and name.strip():
            channel_map[name.strip()] = i
    return channel_map


def build_img_lookup(img_dir: Path, pattern: str = "*.tif") -> Dict[str, Path]:
    imgs = sorted(img_dir.glob(pattern))
    return {p.name: p for p in imgs}


def add_image_paths(samplesheet: pd.DataFrame, img_lookup: Dict[str, Path]) -> pd.DataFrame:
    out = samplesheet.copy()
    out["image_path"] = out["filename"].map(lambda fn: img_lookup.get(fn))
    return out


# Structure segmentation
@dataclass
class StructureSegParams:
    intensity_scaling_param: Tuple[float, float]
    min_area: int
    blur_sigma: float
    log_sigma_1: float
    log_cutoff_1: float
    log_sigma_2: float
    log_cutoff_2: float
    log_sigma_3: float
    log_cutoff_3: float
    vesselness_sigma: Tuple[float, ...]
    vesselness_cutoff: float


def segment_structures_acis_style(
    channel_img: np.ndarray,
    params: StructureSegParams,
) -> np.ndarray:
    """
    ACIS-like segmentation (works for pRab10 objects and lysosomes, just need to adjust values):
    - normal-fit stretch + normalize
    - gaussian blur
    - multi-scale LoG blob detection + frangi vesselness
    - fill holes + remove small objects
    Returns boolean mask (H, W).
    """
    ch = channel_img.astype(float)

    # Normalize to normal distribution stretch
    m, s = norm.fit(ch.flatten())
    stretch_min = max(m - params.intensity_scaling_param[0] * s, float(ch.min()))
    stretch_max = min(m + params.intensity_scaling_param[1] * s, float(ch.max()))
    if stretch_max <= stretch_min:
        # degenerate; return empty
        return np.zeros_like(ch, dtype=bool)

    ch_n = np.clip(ch, stretch_min, stretch_max)
    image_norm = (ch_n - stretch_min) / (stretch_max - stretch_min)

    # Blur
    blurred = filters.gaussian(image_norm, sigma=params.blur_sigma)

    # Blobs via LoG at 3 scales
    log_1 = -1.0 * (params.log_sigma_1**2) * gaussian_laplace(blurred, sigma=params.log_sigma_1)
    log_2 = -1.0 * (params.log_sigma_2**2) * gaussian_laplace(blurred, sigma=params.log_sigma_2)
    log_3 = -1.0 * (params.log_sigma_3**2) * gaussian_laplace(blurred, sigma=params.log_sigma_3)

    log_mask = (log_1 > params.log_cutoff_1) | (log_2 > params.log_cutoff_2) | (log_3 > params.log_cutoff_3)

    # Vessels via frangi
    vesselness = filters.frangi(
        blurred, sigmas=list(params.vesselness_sigma), black_ridges=False
    ) > params.vesselness_cutoff

    combined = log_mask | vesselness
    filled = ndi.binary_fill_holes(combined)
    cleaned = morphology.remove_small_objects(filled, min_size=params.min_area)

    return cleaned.astype(bool)


# Object assignment per cell
def label_objects_within_cells(
    cell_masks: np.ndarray,
    obj_mask: np.ndarray,
) -> Tuple[np.ndarray, Dict[int, int]]:
    """
    For each cell ID in cell_masks, label connected components of obj_mask restricted
    to that cell. Write them into one global label image, and return mapping:
      global_obj_label -> parent_cell_id
    """
    cell_ids = np.unique(cell_masks)
    cell_ids = cell_ids[cell_ids != 0]

    labels_global = np.zeros_like(obj_mask, dtype=np.int32)
    parent_cell: Dict[int, int] = {}
    current_label = 1

    for cid in cell_ids:
        single_cell_mask = (cell_masks == cid)
        obj_in_cell = obj_mask & single_cell_mask
        labeled_in_cell, n = ndi.label(obj_in_cell)

        if n == 0:
            continue

        for ll in np.unique(labeled_in_cell)[1:]:
            labels_global[labeled_in_cell == ll] = current_label
            parent_cell[current_label] = int(cid)
            current_label += 1

    return labels_global, parent_cell


# Colocalization per cell
def compute_cell_coloc(
    cell_masks: np.ndarray,
    lys_int: np.ndarray,
    prab_int: np.ndarray,
    min_pixels: int = 50,
) -> pd.DataFrame:
    cell_ids = np.unique(cell_masks)
    cell_ids = cell_ids[cell_ids != 0]

    rows: List[dict] = []

    for cid in cell_ids:
        m = (cell_masks == cid)
        npx = int(m.sum())
        if npx < min_pixels:
            continue

        a = lys_int[m].astype(np.float64)
        b = prab_int[m].astype(np.float64)

        # Pearson r
        if a.std() == 0 or b.std() == 0:
            pearson_r = np.nan
        else:
            pearson_r = float(np.corrcoef(a, b)[0, 1])

        # per-cell Otsu thresholds
        try:
            tA = float(threshold_otsu(a)) if np.unique(a).size > 1 else 0.0
        except Exception:
            tA = 0.0
        try:
            tB = float(threshold_otsu(b)) if np.unique(b).size > 1 else 0.0
        except Exception:
            tB = 0.0

        a_pos = a > tA
        b_pos = b > tB

        denomA = float(a[a_pos].sum())
        denomB = float(b[b_pos].sum())

        M1 = float(a[a_pos & b_pos].sum() / denomA) if denomA > 0 else np.nan  # lys in prab
        M2 = float(b[b_pos & a_pos].sum() / denomB) if denomB > 0 else np.nan  # prab in lys

        rows.append({
            "cell_id": int(cid),
            "cell_pixels": npx,
            "pearson_r": pearson_r,
            "manders_M1_lys_in_prab": M1,
            "manders_M2_prab_in_lys": M2,
            "threshold_cutoff_lys": tA,
            "threshold_cutoff_prab": tB,
        })

    return pd.DataFrame(rows)


# Cellpose wrappers
@dataclass
class CellposeParams:
    diameter: float = 120
    batch_size: int = 32
    flow_threshold: float = 0.4
    cellprob_threshold: float = 0.0
    tile_norm_blocksize: int = 0  # 0 disables tile norm


def run_cellpose_cells(
    model: models.CellposeModel,
    dapi: np.ndarray,
    gfap: np.ndarray,
    params: CellposeParams,
) -> np.ndarray:
    # stack channels (C, H, W)
    stack = np.stack([dapi, gfap], axis=0)
    masks, flows, styles = model.eval(
        stack,
        batch_size=params.batch_size,
        diameter=params.diameter,
        flow_threshold=params.flow_threshold,
        cellprob_threshold=params.cellprob_threshold,
        normalize={"tile_norm_blocksize": params.tile_norm_blocksize},
    )
    return masks


def run_cellpose_nuclei(
    model: models.CellposeModel,
    dapi: np.ndarray,
    params: CellposeParams,
) -> np.ndarray:
    masks, flows, styles = model.eval(
        dapi,
        batch_size=params.batch_size,
        diameter=params.diameter,
        flow_threshold=params.flow_threshold,
        cellprob_threshold=params.cellprob_threshold,
        normalize={"tile_norm_blocksize": params.tile_norm_blocksize},
    )
    return masks


# Per-image pipeline
@dataclass
class PipelineConfig:
    n_channels: int = 4
    channel_names: Tuple[str, str, str, str] = ("DAPI", "GFAP", "LAMP1", "pRAB10")

    cellpose: CellposeParams = field(default_factory=CellposeParams)

    lys_params: StructureSegParams = field(default_factory=lambda: StructureSegParams(
        intensity_scaling_param=(3, 18),
        min_area=5,
        blur_sigma=1,
        log_sigma_1=3, log_cutoff_1=0.07,
        log_sigma_2=2, log_cutoff_2=0.06,
        log_sigma_3=1, log_cutoff_3=0.04,
        vesselness_sigma=(1,),
        vesselness_cutoff=0.02,
    ))

    prab_params: StructureSegParams = field(default_factory=lambda: StructureSegParams(
        intensity_scaling_param=(1, 9),
        min_area=5,
        blur_sigma=1,
        log_sigma_1=3, log_cutoff_1=0.15,
        log_sigma_2=2, log_cutoff_2=0.13,
        log_sigma_3=1, log_cutoff_3=0.11,
        vesselness_sigma=(1,),
        vesselness_cutoff=0.15,
    ))

    coloc_min_pixels: int = 50

def process_one_image(
    row: pd.Series,
    cfg: PipelineConfig,
    cellpose_model: models.CellposeModel,
) -> Dict[str, pd.DataFrame]:
    """
    Returns dict of dataframes:
      - cell_coloc_df
      - lys_df
      - prab_df
      - (optional) cell_masks/nuc_masks not returned here (keep light)
    """
    img_path = row["image_path"]
    if img_path is None or (isinstance(img_path, float) and np.isnan(img_path)):
        raise FileNotFoundError(f"No image_path for filename={row.get('filename')}")

    img_path = Path(img_path)
    img = tifffile.imread(img_path)

    ch_index = get_channel_indices(row, n_channels=cfg.n_channels)

    # Ensure required channels exist
    for chname in cfg.channel_names:
        if chname not in ch_index:
            raise KeyError(f"Missing channel '{chname}' for image {img_path.name}. Got: {list(ch_index.keys())}")

    dapi = img[ch_index["DAPI"]]
    gfap = img[ch_index["GFAP"]]
    lys  = img[ch_index["LAMP1"]]
    prab = img[ch_index["pRAB10"]]

    # Cellpose segmentations
    cell_masks = run_cellpose_cells(cellpose_model, dapi, gfap, cfg.cellpose)
    nuc_masks  = run_cellpose_nuclei(cellpose_model, dapi, cfg.cellpose)

    # Structure masks (boolean)
    lys_mask  = segment_structures_acis_style(lys, cfg.lys_params)
    prab_mask = segment_structures_acis_style(prab, cfg.prab_params)

    # Label structures within each cell
    lys_labels_global, lys_parent_cell = label_objects_within_cells(cell_masks, lys_mask)
    prab_labels_global, prab_parent_cell = label_objects_within_cells(cell_masks, prab_mask)

    # Regionprops tables
    lys_props = regionprops_table(
        lys_labels_global,
        intensity_image=lys,
        properties=("label", "area", "min_intensity", "mean_intensity", "max_intensity"),
    )
    lys_df = pd.DataFrame(lys_props)
    if not lys_df.empty:
        lys_df["cell_id"] = lys_df["label"].map(lys_parent_cell)
        lys_df.insert(0, "filename", img_path.name)

    prab_props = regionprops_table(
        prab_labels_global,
        intensity_image=prab,
        properties=("label", "area", "min_intensity", "mean_intensity", "max_intensity"),
    )
    prab_df = pd.DataFrame(prab_props)
    if not prab_df.empty:
        prab_df["cell_id"] = prab_df["label"].map(prab_parent_cell)
        prab_df.insert(0, "filename", img_path.name)

    # Colocalization per cell
    cell_coloc_df = compute_cell_coloc(cell_masks, lys, prab, min_pixels=cfg.coloc_min_pixels)
    cell_coloc_df.insert(0, "filename", img_path.name)

    return {
        "cell_coloc_df": cell_coloc_df,
        "lys_df": lys_df,
        "prab_df": prab_df,
        # if you later want nucleus stats, add them here
    }


# Batch runner
def run_on_folder(
    samplesheet_csv: Path,
    img_dir: Path,
    out_dir: Path,
    pattern: str = "*.tif",
    use_gpu: bool = True,
    cfg: Optional[PipelineConfig] = None,
) -> Dict[str, Path]:
    cfg = cfg or PipelineConfig()
    out_dir.mkdir(parents=True, exist_ok=True)

    samplesheet = pd.read_csv(samplesheet_csv)

    img_lookup = build_img_lookup(img_dir, pattern=pattern)
    samplesheet = add_image_paths(samplesheet, img_lookup)

    # Keep only rows with images present
    present = samplesheet["image_path"].notna()
    missing = samplesheet.loc[~present, "filename"].tolist()
    if missing:
        print(f"[WARN] {len(missing)} filenames not found in folder; skipping first few: {missing[:5]}")
    samplesheet = samplesheet.loc[present].reset_index(drop=True)

    # Init model once
    cellpose_model = models.CellposeModel(gpu=use_gpu)

    coloc_all = []
    lys_all = []
    prab_all = []

    failures = []

    for i, row in samplesheet.iterrows():
        try:
            res = process_one_image(row, cfg, cellpose_model)

            if not res["cell_coloc_df"].empty:
                coloc_all.append(res["cell_coloc_df"])
            if not res["lys_df"].empty:
                lys_all.append(res["lys_df"])
            if not res["prab_df"].empty:
                prab_all.append(res["prab_df"])

        except Exception as e:
            tb = traceback.format_exc()
            failures.append((fname, type(e).__name__, str(e), tb))
            print(f"[FAIL] {fname}: {type(e).__name__}: {e}")
            continue

        if (i + 1) % 10 == 0:
            print(f"Processed {i+1}/{len(samplesheet)} images so far")

    # Concatenate + write
    out_paths: Dict[str, Path] = {}

    coloc_all_df = pd.concat(coloc_all, ignore_index=True) if coloc_all else pd.DataFrame()
    lys_all_df   = pd.concat(lys_all, ignore_index=True) if lys_all else pd.DataFrame()
    prab_all_df  = pd.concat(prab_all, ignore_index=True) if prab_all else pd.DataFrame()

    out_paths["cells_coloc_all"] = out_dir / "cells_coloc_all.csv"
    out_paths["lys_objects_all"] = out_dir / "lys_objects_all.csv"
    out_paths["prab_objects_all"] = out_dir / "prab_objects_all.csv"

    coloc_all_df.to_csv(out_paths["cells_coloc_all"], index=False)
    lys_all_df.to_csv(out_paths["lys_objects_all"], index=False)
    prab_all_df.to_csv(out_paths["prab_objects_all"], index=False)

    if failures:
        fail_path = out_dir / "failures.csv"
        pd.DataFrame(failures, columns=["filename", "error"]).to_csv(fail_path, index=False)
        out_paths["failures"] = fail_path
        print(f"[WARN] {len(failures)} failures written to {fail_path}")

    print("Done.")
    return out_paths

In [2]:
# Example call

if __name__ == "__main__":
    samplesheet_csv = Path("/Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Hannah_RepresentativeImages/hb_samplesheet.csv")
    img_dir = Path("/Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Hannah_RepresentativeImages")
    out_dir = img_dir / "pipeline_outputs"

    paths = run_on_folder(samplesheet_csv, img_dir, out_dir, pattern="*.tif", use_gpu=True)
    for k, v in paths.items():
        print(k, "->", v)


Processed 10/16
Done.
cells_coloc_all -> /Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Hannah_RepresentativeImages/pipeline_outputs/cells_coloc_all.csv
lys_objects_all -> /Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Hannah_RepresentativeImages/pipeline_outputs/lys_objects_all.csv
prab_objects_all -> /Users/kelpschdj/Documents/DataTecnica/Hannah_Bailey/Hannah_RepresentativeImages/pipeline_outputs/prab_objects_all.csv
